# MedScope step 2: build SLAKE counterfactual views

This notebook downloads SLAKE, keeps the English abnormality questions that have lesion masks, builds the four views per case (original, lesion removed, matched random edit, degraded input) with LaMa inpainting, runs the automated quality checks, and saves everything to Google Drive.

**Runtime:** set *Runtime > Change runtime type > T4 GPU* (LaMa runs much faster on a GPU).

## 1. Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/medscope/slake'   # final outputs land here
WORK = '/content/work'                                 # fast local disk for building
RAW = f'{WORK}/raw/slake'
!mkdir -p {DRIVE_OUT} {RAW}

## 2. Get the code
If the repo is private, add a GitHub token as a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar) with read access to the repo.

In [ ]:
import os
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@github.com/bhaskargurram-ai/medscope' if token else 'https://github.com/bhaskargurram-ai/medscope'
!rm -rf /content/medscope && git clone -q {url} /content/medscope
%cd /content/medscope
!pip install -q -e . simple-lama-inpainting huggingface_hub
!python -m unittest 2>&1 | tail -3

## 3. Download SLAKE from Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, shutil, pathlib

for f in ['train.json', 'validation.json', 'test.json', 'mask.txt', 'imgs.zip']:
    p = hf_hub_download('BoKelvin/SLAKE', f, repo_type='dataset')
    shutil.copy(p, f'{RAW}/{f}')

with zipfile.ZipFile(f'{RAW}/imgs.zip') as z:
    z.extractall(RAW)
# make sure images end up at RAW/imgs/xmlab*/
raw = pathlib.Path(RAW)
if not (raw / 'imgs').exists():
    found = next(raw.rglob('xmlab*')).parent
    found.rename(raw / 'imgs')
print(len(list((raw / 'imgs').glob('xmlab*'))), 'image folders')

## 4. Check the files match what the loader expects
Look at the output before moving on: question fields should include `img_name`, `q_lang` and `content_type`; `mask.txt` should list ids with names (some with *Cancer*, *Tumor* or *Edema*); and the mask should contain a few small integer ids.

In [ ]:
import json, numpy as np
from PIL import Image
q = json.load(open(f'{RAW}/train.json'))
print('questions in train:', len(q))
print('fields:', sorted(q[0].keys()))
print('content types:', sorted({x.get('content_type') for x in q}))
print(open(f'{RAW}/mask.txt').read()[:600])
m = np.asarray(Image.open(f'{RAW}/imgs/' + q[0]['img_name'].replace('source.jpg', 'mask.png')))
print('mask shape', m.shape, 'ids', np.unique(m)[:20])

## 5. Turn SLAKE into case manifests

In [ ]:
!python scripts/prepare_slake.py --root {RAW} --out {WORK}/cases

## 6. Build the four views with LaMa inpainting

In [ ]:
for split in ['train', 'validation', 'test']:
    !python scripts/build_views.py --manifest {WORK}/cases/cases_{split}.jsonl --out {WORK}/views/{split} --inpainter lama

## 7. Automated quality checks
Test and validation cases are also checked against training images for duplicates. Look for a shortcut AUC at or below 0.6; above that, the edits are detectable from texture alone.

In [ ]:
!python scripts/check_views.py --views {WORK}/views/train/views.jsonl
!python scripts/check_views.py --views {WORK}/views/validation/views.jsonl --dedup-against {WORK}/views/train/views.jsonl
!python scripts/check_views.py --views {WORK}/views/test/views.jsonl --dedup-against {WORK}/views/train/views.jsonl

## 8. Look at a few examples

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict
views = [json.loads(l) for l in open(f'{WORK}/views/test/views_clean.jsonl')]
by_case = defaultdict(dict)
for v in views:
    by_case[v['case_id']][v['kind']] = v
kinds = ['original', 'target_removal', 'random_removal', 'degraded']
cases = list(by_case)[:4]
fig, axes = plt.subplots(len(cases), 4, figsize=(12, 3 * len(cases)))
for r, cid in enumerate(cases):
    for c, k in enumerate(kinds):
        axes[r, c].imshow(Image.open(by_case[cid][k]['image_path']))
        axes[r, c].set_title(k if r == 0 else '')
        axes[r, c].axis('off')
    axes[r, 0].set_ylabel(by_case[cid][k]['question'][:40])
plt.tight_layout()

## 9. Save to Drive

In [ ]:
!cp -r {WORK}/cases {WORK}/views {DRIVE_OUT}/
!du -sh {DRIVE_OUT}
print('saved to', DRIVE_OUT)

**Note:** view manifests store absolute paths under `/content/work`. A later notebook will copy the folder back to that path (or rewrite the paths) before training.